## Generating Embeddings using SentenceTransformer

Until now, we have discussed the concept of embeddings.

Instead of manually assigning vectors, modern NLP models use pretrained transformer models to generate meaningful embeddings automatically.

**SentenceTransformer** converts an entire sentence into a dense vector while preserving its semantic meaning.

These embeddings can then be compared using **Cosine Similarity**.


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")


`all-MiniLM-L6-v2` is a lightweight pretrained model that converts each sentence into a **384-dimensional embedding vector**.

In [ ]:
sentences = [
    "I love machine learning.",
    "Artificial Intelligence is fascinating.",
    "I enjoy playing cricket."
]

embeddings = model.encode(sentences)

print("Embedding shape:", embeddings.shape)


In [ ]:
similarity = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:")
print(similarity)


### Observation

- The first two sentences are semantically related, so they have a **higher cosine similarity**.
- The cricket sentence is unrelated, so its similarity with the other two is much lower.

This demonstrates how pretrained sentence embeddings capture **semantic meaning**, not just exact word matches.


# Session 7 — From Similarity to Sequence Models
**Instructor: Soham Bhattacharya**  
Date: June 24, 2026 | ~12:25 PM – 1:34 PM

---

### Topics Covered
1. Cosine Similarity & Similarity Matrices (recap/interactive)
2. Word Embeddings & the Embedding Projector (projector.tensorflow.org)
3. Tokenization — OpenAI Tokenizer demo
4. Manual Tokenization in Python
5. Bag-of-Words (BoW) Sentiment Analysis
6. Negation Handling in Sentiment
7. ELIZA — Rule-Based Chatbot (historical context)
8. Why Normal Models Are Not Enough → Sequence Models
9. Recurrent Neural Networks (RNN) — Introduction
10. LSTM Networks — Colah's blog overview

---
## Part 1 — Cosine Similarity & Similarity Matrix

**Concept**: Cosine similarity measures how similar two vectors are by computing the cosine of the angle between them. A value of 1 means identical direction (perfectly similar), 0 means orthogonal (unrelated).

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

We can compute a **similarity matrix** for a group of students (or words) to see who/what is most similar to whom/what.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Example: each student represented by a feature vector
# Features could be: [study_hours, assignments_done, quiz_score, attendance]
student_vectors = {
    'Aarav':  [0.9, 0.8, 0.95, 1.0],
    'Riya':   [0.85, 0.9, 0.88, 0.95],
    'Kabir':  [0.7, 0.75, 0.80, 0.85],
    'Meera':  [0.6, 0.65, 0.72, 0.80],
    'Dev':    [0.88, 0.82, 0.90, 0.92],
    'Sara':   [0.95, 0.88, 0.99, 0.98],
}

students = list(student_vectors.keys())
vectors = np.array(list(student_vectors.values()))

# Compute cosine similarity matrix
similarity_df = pd.DataFrame(
    cosine_similarity(vectors),
    index=students,
    columns=students
)

print("Similarity Matrix:")
print(similarity_df.round(6))

# Who is most similar to Sara?
sara_similarities = similarity_df['Sara'].drop('Sara').sort_values(ascending=False)
print("\nMost similar to Sara:", sara_similarities.idxmax())

---
## Part 2 — Word Embeddings

**What is an Embedding?**  
An embedding is a dense vector of real numbers that represents a word (or sentence, image, etc.).  
Words with similar meanings cluster close together in embedding space.

- In **RAG systems**, we convert documents into embeddings
- In **recommendation systems**, we convert products/users into embeddings
- In **NLP**, we convert words into embeddings

**Key Tool**: [Embedding Projector (projector.tensorflow.org)](https://projector.tensorflow.org/)  
- Uses Word2Vec 10K vocabulary
- 10,000 points in 200 dimensions
- Visualized using PCA (Principal Component Analysis) to reduce to 3D
- PCA is approximate — total variance described: ~8.5%

**Demonstration in class**:  
- Searched for word "school" → nearest neighbors: college, university, education, attended, student, classroom, graduate, teacher
- Selected 1000 points around "school" → highlighted cluster
- **Distance metrics used**: Cosine or Euclidean
- **Visualization methods**: UMAP, T-SNE, PCA, Custom

In [ ]:
# Demonstrating embedding concept with random vectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# In real Word2Vec embeddings, each word is a 300-dim vector
# Here we simulate with small 5-dim vectors for illustration
# In RAG systems we convert documents into embeddings

# Simulated word embeddings (in practice, loaded from Word2Vec/GloVe)
word_embeddings = {
    'school':     [0.9, 0.8, 0.3, 0.1, 0.7],
    'college':    [0.85, 0.75, 0.35, 0.15, 0.65],
    'university': [0.80, 0.70, 0.40, 0.20, 0.60],
    'education':  [0.75, 0.65, 0.45, 0.25, 0.55],
    'hospital':   [0.2, 0.3, 0.9, 0.8, 0.1],
    'doctor':     [0.15, 0.25, 0.85, 0.75, 0.05],
}

words = list(word_embeddings.keys())
vecs = np.array(list(word_embeddings.values()))

sim = cosine_similarity(vecs)
df = pd.DataFrame(sim, index=words, columns=words)

print("Word similarity matrix (simulated embeddings):")
print(df.round(4))

print("\nNearest neighbors to 'school':")
school_sim = df['school'].drop('school').sort_values(ascending=False)
print(school_sim)

---
## Part 3 — Tokenization

**Definition**: Tokenization is the process of breaking text into smaller units called **tokens** before sending it to an AI model.

**Key Rules (from OpenAI Tokenizer)**:
- 1 token ≈ 4 characters of text
- 100 tokens ≈ 75 words
- Tokens can be words, sub-words, or individual characters
- Common words = 1 token; rare words can be split into multiple tokens
- Even Hindi/non-English text is tokenized (e.g., `नमस्ते, मुझे AI सीखना है` = 11 tokens, 25 characters)

**Tool**: [OpenAI Tokenizer](https://platform.openai.com/tokenizer)

**Example shown in class**:  
`नमस्ते, मुझे AI सीखना है` → 11 tokens, 25 characters  
Token IDs: `[976, 8249, 673, 625, 1899, 1899, 13]` (varies by sentence)

**Q&A from class**:
- Q: Does it split on spaces only? A: No — it also splits sub-words. "Namaste" → "Na" and "ma" could be separate tokens.
- Q: Is "Mujhe" a token? A: No — it's too rare/specific, split into syllabic sub-tokens
- Q: Do "and"/"or" become tokens? A: Yes — common short words are usually single tokens

In [ ]:
# Text → Tokens → Token IDs → Embeddings → Model
# Manual tokenization pipeline (simplified, whitespace-based)

sentence = "The movie was not good"

# Step 1: Tokenize (split into words)
tokens = sentence.lower().split()
print("Tokens:", tokens)

# Step 2: Build vocabulary (word → integer ID)
vocab = {
    'the': 1,
    'movie': 2,
    'was': 3,
    'not': 4,
    'good': 5
}

# Step 3: Convert tokens to IDs
token_ids = [vocab[word] for word in tokens]
print("Token IDs:", token_ids)

# Output: [1, 2, 3, 4, 5]
# These IDs are then looked up in an embedding table to get vectors

---
## Part 4 — Bag-of-Words (BoW) Sentiment Analysis

**Bag-of-Words** treats text as an unordered collection of words (ignores order/structure).  
We can use a simple word-count approach for sentiment analysis.

**Approach**: 
- Maintain a list of positive words and negative words
- Count how many positive/negative words appear in the text
- If positive_count > negative_count → Positive sentiment

**Limitation demonstrated in class**:
- "Dog bites man" vs "Man bites dog" → same BoW representation! Context is lost.
- This is the core weakness of BoW.

In [ ]:
# Simple Bag-of-Words Sentiment Analysis

positive_words = ['good', 'great', 'amazing', 'excellent', 'love']
negative_words = ['bad', 'terrible', 'poor', 'hate', 'boring']

def simple_sentiment(sentence):
    words = sentence.lower().replace(',', '').split()
    score = 0
    for word in words:
        if word in positive_words:
            score += 1
        elif word in negative_words:
            score -= 1
    
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

# Test sentences from class
test_sentences = [
    "The movie was good",
    "#The movie was not good",   # Note: 'not' is ignored in basic BoW!
    "the movie was good not",    # Same words, wrong sentiment
]

for s in test_sentences:
    print(f"{s!r} → {simple_sentiment(s)}")

print()
# Demonstrates the BoW problem: word order doesn't matter
print("BoW Problem demo:")
print(simple_sentiment("The movie was good"))    # Positive (correct)
print(simple_sentiment("The movie was not bad")) # Positive (WRONG! negation ignored)
print(simple_sentiment("The movie was bad not")) # Negative (wrong order, same result)

---
## Part 5 — Handling Negation in Sentiment

**Problem**: Simple BoW misclassifies "not good" as positive because it counts "good" independently.

**Solution**: Check if the previous word was "not" → flip the effect of the following word.

In [ ]:
# Improved: Handle negation with context

positive_words = ['good', 'great', 'amazing', 'excellent', 'love']
negative_words = ['bad', 'terrible', 'poor', 'hate', 'boring']

def better_sentiment_with_not(sentence):
    words = sentence.lower().replace(',', '').split()
    score = 0
    
    for i, word in enumerate(words):
        previous_word = words[i - 1] if i > 0 else ''
        
        if word in positive_words:
            if previous_word == 'not':
                score -= 1  # "not good" → negative
            else:
                score += 1
        elif word in negative_words:
            if previous_word == 'not':
                score += 1  # "not bad" → positive
            else:
                score -= 1
    
    if score > 0:
        return "Positive"
    elif score < 0:
        return "Negative"
    else:
        return "Neutral"

# Richer test set shown in class
test_cases = [
    "The movie was good",         # Positive
    "The movie was not good",     # Negative (negation handled)
    "The movie was not bad",      # Positive (negation handled)
    "The movie was good not",     # Still positive (word order edge case)
]

for s in test_cases:
    print(f"{s!r:35} → {better_sentiment_with_not(s)}")

print()
# Word order problem in BoW:
print("Word Order Problem (BoW ignores sequence):")
s1 = "I only said she stole my phone"
s2 = "Only I said she stole my phone"
s3 = "I said only she stole my phone"
s4 = "I said she only stole my phone"
print(f"S1: {s1!r} → {simple_sentiment(s1)}")
print(f"S2: {s2!r} → {simple_sentiment(s2)}")
print("Both produce the SAME result — BoW ignores word order!")

In [ ]:
# Processing multiple sentences — tokenization

sentences = [
    "dog bites man",
    "man bites dog",
    "the movie was good",
    "the movie was not good"
]

for sentence in sentences:
    print(f"{sentence!r:30} -> {sentence.split()}")

print()
print("BoW representation ignores order:")
print("'dog bites man' ==? 'man bites dog' in BoW:",
      sorted("dog bites man".split()) == sorted("man bites dog".split()))

---
## Part 6 — ELIZA: The First Chatbot (Historical)

**ELIZA** was a natural language conversation program created by **Joseph Weizenbaum in January 1966** at MIT.

- It simulates a Rogerian psychotherapist
- Uses **pattern matching and substitution rules** — no understanding
- Famous for making people believe it understood them (**ELIZA effect**)

**Demo**: [masswerk.at/elizabot/](https://www.masswerk.at/elizabot/)

**Example exchange shown in class**:
```
ELIZA: How do you do. Please tell me your problem.
YOU:   I have a problem of students not answering polls
ELIZA: Can you elaborate on that?
YOU:   I give a poll they take so much time
```

**Why it matters**: ELIZA is a rule-based system — the exact opposite of learned embeddings. It shows how far we've come.

**Key Insight**: ELIZA used keyword matching rules like:
- If user says "I feel X" → respond "Why do you feel X?"
- If user mentions "problem" → respond with elaboration request

In [ ]:
# Simplified ELIZA-like rule-based chatbot
import re

def eliza_response(user_input):
    """Super-simplified ELIZA: pattern-matching responses."""
    user_input = user_input.lower()
    
    rules = [
        (r'i feel (.*)', lambda m: f"Why do you feel {m.group(1)}?"),
        (r'i have a problem (.*)', lambda m: f"Can you elaborate on your problem {m.group(1)}?"),
        (r'i (.*)', lambda m: f"You say you {m.group(1)}. Can you elaborate?"),
        (r'why (.*)', lambda m: f"What do you think about why {m.group(1)}?"),
        (r'(.*)', lambda m: "Please tell me more about that."),
    ]
    
    for pattern, response_fn in rules:
        match = re.match(pattern, user_input)
        if match:
            return response_fn(match)
    
    return "Can you elaborate on that?"

# Demo
test_inputs = [
    "I feel stressed about exams",
    "I have a problem with students not answering polls",
    "I think AI is amazing",
    "Why is the sky blue?"
]

for inp in test_inputs:
    print(f"YOU  : {inp}")
    print(f"ELIZA: {eliza_response(inp)}")
    print()

---
## Part 7 — Why Normal Models Are Not Enough

**Architectural Comparison** (slide shown at 1:21 PM)

### Bag-of-Words Model
- Counts overall frequencies but **discards sentence structure entirely**
- The sentence is reduced to an unordered container of independent tokens
- **Problem**: "not good" and "good not" are treated identically

### Sequence Model
- Reads tokens **sequentially**
- Captures modifier relationships (e.g., "not + good" combining into a **negative sentiment** block)

**Applied Rule**: Bag-of-words ignores word order, which can cause key context like **negation to be lost**. Sequence models read word sequences in order.

**Word order demonstration (class exercise)**:
```
Sentence:               Key meaning changes:
"I ONLY said she stole my phone"   → I said it, not others
"Only I SAID she stole my phone"   → I said it, didn't do it  
"I said ONLY she stole my phone"   → nobody else did
"I said she ONLY stole my phone"   → didn't do worse
```
Same words — 4 completely different meanings. BoW can't distinguish these!

In [ ]:
# The famous word-order demonstration from class

sentences = [
    "I only said she stole my phone",
    "Only I said she stole my phone",
    "I said only she stole my phone",
    "I said she only stole my phone",
]

for s in sentences:
    print(s)

print()
print("BoW representations (sorted word bags):")
for s in sentences:
    print(f"  {sorted(s.lower().split())}")

print()
print("→ All 4 sentences have IDENTICAL BoW representations!")
print("→ Yet they carry 4 DIFFERENT meanings.")
print("→ This is why we need sequence-aware models (RNN, LSTM, Transformer).")

---
## Part 8 — Recurrent Neural Networks (RNN)

**Definition**: RNNs are neural networks with **loops** in them, allowing information to persist.  
Unlike feedforward networks, RNNs process sequences step-by-step, maintaining a **hidden state** that passes information from one step to the next.

**Human analogy (from class)**:  
"As you read this essay, you understand each word based on your understanding of *previous words*. You don't throw everything away and start thinking from scratch again. Your thoughts have persistence."

**Traditional NNs can't do this** — they process each input independently.

**Key Architecture**:
```
Input:  x₁  x₂  x₃  x₄  ...  xₜ
         ↓   ↓   ↓   ↓        ↓
Hidden: h₁ → h₂ → h₃ → h₄ → hₜ
         ↓   ↓   ↓   ↓        ↓
Output: y₁  y₂  y₃  y₄  ...  yₜ
```
Each hidden state `hₜ = f(hₜ₋₁, xₜ)` depends on the current input AND the previous hidden state.

**Source**: [Colah's blog — Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) (August 27, 2015)

**RNN Problem**: Suffers from **vanishing gradient** — struggles to learn long-range dependencies.

In [ ]:
# Conceptual RNN implementation (forward pass only)
import numpy as np

class SimpleRNN:
    """Minimal RNN to show the concept of hidden state persistence."""
    
    def __init__(self, input_size, hidden_size):
        # Weight matrices (randomly initialized)
        np.random.seed(42)
        self.Wx = np.random.randn(hidden_size, input_size) * 0.1   # input → hidden
        self.Wh = np.random.randn(hidden_size, hidden_size) * 0.1  # hidden → hidden
        self.b  = np.zeros((hidden_size, 1))                        # bias
        self.hidden_size = hidden_size
    
    def forward(self, inputs):
        """
        inputs: list of input vectors (each shape: [input_size, 1])
        Returns: list of hidden states for each time step
        """
        h = np.zeros((self.hidden_size, 1))  # Initial hidden state = 0
        hidden_states = []
        
        for x in inputs:
            # RNN update rule: h_t = tanh(Wx * x_t + Wh * h_{t-1} + b)
            h = np.tanh(self.Wx @ x + self.Wh @ h + self.b)
            hidden_states.append(h.copy())
            print(f"  Hidden state norm: {np.linalg.norm(h):.4f}")
        
        return hidden_states

# Example: processing a 3-word sentence
rnn = SimpleRNN(input_size=4, hidden_size=3)

# "The movie was good" → 4 tokens, each represented as a 4-dim vector
# (In practice, these come from an embedding lookup)
word_vectors = [
    np.array([[0.1], [0.2], [0.3], [0.4]]),  # 'the'
    np.array([[0.5], [0.1], [0.2], [0.3]]),  # 'movie'
    np.array([[0.2], [0.3], [0.1], [0.5]]),  # 'was'
    np.array([[0.8], [0.9], [0.1], [0.2]]),  # 'good'
]

print("Processing 'The movie was good' through RNN:")
hidden_states = rnn.forward(word_vectors)
print(f"Final hidden state (summary of the sentence): {hidden_states[-1].T}")

---
## Part 9 — LSTM Networks

**LSTM = Long Short-Term Memory**  
Introduced to solve the **vanishing gradient problem** of vanilla RNNs.

**Core Idea**: LSTMs have a **cell state** (long-term memory) in addition to the hidden state (short-term memory). They use **gates** to control what information is kept, forgotten, or outputted.

### Three Gates in LSTM:

| Gate | Symbol | Purpose |
|------|---------|----------|
| **Forget Gate** | f | Decides what to throw away from cell state |
| **Input Gate** | i | Decides what new info to store in cell state |
| **Output Gate** | o | Decides what to output based on cell state |

### LSTM Update Rules:
```
f_t = σ(W_f · [h_{t-1}, x_t] + b_f)          # Forget gate
i_t = σ(W_i · [h_{t-1}, x_t] + b_i)          # Input gate
C̃_t = tanh(W_C · [h_{t-1}, x_t] + b_C)       # Candidate cell state
C_t = f_t * C_{t-1} + i_t * C̃_t              # Update cell state
o_t = σ(W_o · [h_{t-1}, x_t] + b_o)          # Output gate
h_t = o_t * tanh(C_t)                         # New hidden state
```

**Key difference from RNN**: The cell state `C_t` runs straight through the entire chain with only minor linear interactions — information can flow unchanged across many steps.

**Resource**: [Colah's blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)

In [ ]:
# LSTM using PyTorch (or show the concept manually)
# Using PyTorch for a proper LSTM demonstration

try:
    import torch
    import torch.nn as nn

    # Simple LSTM for sentiment analysis
    class SentimentLSTM(nn.Module):
        def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embed_dim)
            self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
            self.fc = nn.Linear(hidden_dim, output_dim)
            self.sigmoid = nn.Sigmoid()
        
        def forward(self, text):
            embedded = self.embedding(text)        # [batch, seq, embed_dim]
            output, (hidden, cell) = self.lstm(embedded)  # LSTM processes sequence
            # Use final hidden state for classification
            prediction = self.sigmoid(self.fc(hidden.squeeze(0)))
            return prediction

    # Quick demo
    model = SentimentLSTM(vocab_size=100, embed_dim=8, hidden_dim=16, output_dim=1)
    
    # Fake token IDs for "the movie was good" = [1, 2, 3, 4]
    sample_input = torch.tensor([[1, 2, 3, 4]])  # shape: [1, 4]
    output = model(sample_input)
    print(f"LSTM model output (sentiment probability): {output.item():.4f}")
    print("→ Probability > 0.5 = Positive, < 0.5 = Negative")
    print()
    print("LSTM Model Architecture:")
    print(model)

except ImportError:
    print("PyTorch not available. Here's the concept:")
    print("LSTM(input) → embedding → LSTM layer → linear layer → sigmoid → sentiment")
    print("The LSTM layer maintains both hidden state (h) and cell state (C)")

---
## Summary: Evolution of Text Models

| Model | Handles Order? | Handles Context? | Example Use |
|-------|---------------|-----------------|-------------|
| **BoW** | ❌ No | ❌ No | Basic keyword search |
| **ELIZA (Rule-based)** | ✅ Pattern-based | ❌ No | Old chatbots |
| **RNN** | ✅ Yes | ✅ Short-range | Text generation |
| **LSTM** | ✅ Yes | ✅ Long-range | Translation, sentiment |
| **Transformer** | ✅ Yes | ✅ Full context | GPT, BERT, Claude |

**Pipeline shown in class**:
```
Text → Tokenization → Token IDs → Embedding Lookup → Model (RNN/LSTM/Transformer)
```

**Key Resources from teacher**:
- [Embedding Projector](https://projector.tensorflow.org/)
- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [ELIZA demo](https://www.masswerk.at/elizabot/)
- [Colah's LSTM blog](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)